# Medical Dictation → SOAP Note

End-to-end pipeline:

1. **Part A – Speech-to-text:** local Whisper (`faster-whisper` or `sherpa-onnx`) with VAD chunking and medical-vocabulary biasing. Audio never leaves the machine.
2. **Part B – Text-to-SOAP:** LLM (local Ollama by default; OpenAI / Anthropic optional) with a strict, schema-constrained prompt + few-shot example.
3. **Bonus – Validation:** rule-based + LLM-as-judge check that no *Subjective* content sits in *Objective*, and a grounding check that every number in the note exists in the transcript.

All logic lives in the `soap_pipeline/` package; this notebook just drives it.

In [1]:
import json, os, sys
sys.path.insert(0, os.path.abspath('..'))   # make the package importable from notebooks/
os.chdir(os.path.abspath('..'))

from soap_pipeline.audio import load_audio, duration_seconds
from soap_pipeline.transcribe import transcribe
from soap_pipeline.medical_terms import fix_medical_terms
from soap_pipeline.soap import generate_soap, to_markdown
from IPython.display import Markdown, display

AUDIO = 'data/sample_dictation.mp3'
ASR_BACKEND = 'faster-whisper'   # or 'sherpa-onnx' (run scripts/download_sherpa_models.sh first)
LLM_PROVIDER = os.getenv('SOAP_LLM_PROVIDER', 'ollama')  # or 'openai' / 'anthropic' (set OPENAI_API_KEY / ANTHROPIC_API_KEY)
RUN_ASR = False                  # True = re-transcribe (~15 min on a 2-core CPU); False = reuse outputs/

## 1. Load audio

In [2]:
samples = load_audio(AUDIO)
print(f'{duration_seconds(samples):.1f} s of 16 kHz mono audio, {len(samples):,} samples')

825.0 s of 16 kHz mono audio, 13,200,258 samples


## 2. Part A – Transcribe (verbatim)
`temperature=0`, beam search, Silero VAD splitting on pauses (Whisper has a 30 s window), and an `initial_prompt` of medical terms to bias decoding. A small, auditable correction map fixes a handful of observed mis-hearings; the untouched output is saved too.

In [3]:
if RUN_ASR:
    tr = transcribe(AUDIO, backend=ASR_BACKEND, model='medium.en', postprocess=False)
    open('outputs/transcript_raw.txt', 'w').write(tr.text + '\n')
    transcript = fix_medical_terms(tr.text)
    open('outputs/transcript.txt', 'w').write(transcript + '\n')
    print(f'{tr.backend} / {tr.model}: {tr.audio_seconds:.0f}s audio in {tr.elapsed_seconds:.0f}s')
else:
    transcript = open('outputs/transcript.txt').read().strip()
print(transcript[:1500], '...')

Since you win Hi, I've had this pain on the outside of my... uh... right elbow now created it I first started noticing it several months ago, but... Recently, it's just been more painful. Okay. So you said several months ago, did anything happen several months ago? Was there any sort of trigger, trauma? anything like that to that area. No, there wasn't any any trauma or any triggers that that I noticed. I was just Feeling it. Alright. a bit more at the end of work. I Yeah, I was just feeling the pain a bit more at the end of work. Okay, does anything make it better or worse, the pain? uh... Yeah, really if I'm just resting the elbow. It makes it better and I've tried. uh... things like ibuprofen which has helped with the pain, I'll do that for. Um... hoping I get through. work sometimes if the the pain is bad enough Right, okay. Um... And if you were to describe the quality of the pain, is it sharp, throbby, achy? I Mmm uh... punish you Well, it's achy and then sometimes depending on t

## 3. Part B – Generate the SOAP note

In [4]:
note = generate_soap(transcript, engine='llm', provider=LLM_PROVIDER)
print('engine:', note['_meta']['engine'])
display(Markdown(to_markdown(note)))

engine: llm:anthropic:claude-sonnet-5-5


# SOAP Note

**Patient:** sex: female

## S - Subjective
**Chief complaint:** Right lateral elbow pain

**History of present illness**
- Pain on outside of right elbow, first noticed several months ago
- No trauma or trigger identified
- Recently more painful; progressively worsening over several months
- Pain worse at end of work
- Pain quality achy, sometimes sharp with movement
- Pain severity 4/10
- Aggravated by bending wrist down, picking up or holding heavy objects, and repetitive line work
- Rest improves pain
- Ibuprofen helps; takes it at times to get through work
- Drugstore topical product took the edge off slightly
- No previous episodes of similar pain
- Pain localized to one spot, no radiation
- Reports full ROM of right elbow and hand
- Able to manage daily tasks and work so far, but becoming an issue at work
- Right-hand dominant

**Past medical / surgical history**
- Otherwise healthy, no long-standing medical conditions
- C-section 17 years ago (non-emergency)
- Immunizations up to date

**Medications**
- Tylenol for pain
- Advil (ibuprofen) for pain

**Allergies**
- No known allergies

**Social / family history**
- Works on a line with repetitive tasks
- Never smoked
- Denies marijuana use
- Alcohol about 1 drink per week
- No recent sick contacts
- No recent travel
- No family history of musculoskeletal issues
- No family history of neurological issues
- No family history of autoimmune conditions

**Review of systems**
- Denies numbness or tingling in arm
- Denies arm coldness
- Denies pain in other joints or muscle groups
- Denies visible redness or swelling of right elbow
- Denies misalignment or asymmetry compared to other elbow
- Denies skin changes or deformity
- Denies warmth over area
- Denies weight loss, weight stable
- Denies fever or chills
- Denies fatigue, energy good
- Denies rashes
- Denies changes in vision, hearing, smell or taste
- Denies cough, shortness of breath or wheezing
- Denies chest pain, palpitations or dizziness
- Denies change in bowel movements
- Denies nausea or vomiting
- Denies urinary symptoms
- Denies memory loss or confusion

## O - Objective
**Vital signs**
- _Not documented_

**Physical exam**
- Pain on palpation of lateral right elbow
- No pain on palpation of medial elbow
- No pain on palpation of forearm or biceps
- Pain with wrist extension with elbow in full extension
- Pain with wrist flexion with elbow in full extension
- No pain with elbow flexion
- No pain with shoulder extension
- No pain with shoulder adduction
- No pain with forearm pronation and supination

**Labs / imaging**
- _Not documented_

## A - Assessment
- Likely lateral epicondylitis (tennis elbow), right elbow
- Possible predisposing factor: repetitive strain at work with continuous contraction of wrist extensor muscle group

## P - Plan
- No imaging indicated: no trauma, low concern for fracture
- Activity modification at work: avoid repetitive tasks, especially those using forearm extensor muscles
- Compression sleeve
- Splint optional
- Continue Tylenol or NSAIDs for pain
- Physical therapy to improve mobility and strength of arm
- Apply ice for pain and swelling
- Education: typically resolves in a couple of months with PT and compression; up to 2 years at most

---
_Engine: llm:anthropic:claude-sonnet-5-5 | Subjective items moved out of Objective: 0 | Ungrounded numbers: none_

## 4. Bonus – Validation report

In [5]:
print(json.dumps(note['_meta']['validation'], indent=2))

{
  "subjective_in_objective_moved": [],
  "ungrounded_numbers": [],
  "passed": true
}


In [6]:
json.dump(note, open('outputs/soap_note.json', 'w'), indent=2, ensure_ascii=False)
open('outputs/soap_note.md', 'w').write(to_markdown(note) + '\n')
print('saved outputs/soap_note.json and outputs/soap_note.md')

saved outputs/soap_note.json and outputs/soap_note.md
